# CMI-Bench · From a music question to a benchmark score

An audio language model can describe a song in words. Music information retrieval (MIR) also asks it to return things we can check precisely, such as the key of a recording or the times of its beats. **CMI-Bench**, a benchmark for music instruction following, presents these tasks as natural-language instructions paired with audio and scores the model's answers against dataset annotations.

In this tutorial you will inspect the benchmark inputs, run Qwen2-Audio and a trained MIR specialist on each clip, and score their predictions with the official evaluator.

| Task | Dataset | What the model returns | What the evaluator measures |
|---|---|---|---|
| Key detection | GiantSteps | A key label such as `D major` | How closely the predicted key matches the annotated key |
| Beat tracking | Ballroom | Timestamps such as `0.58s, 1.23s, …` | Agreement with the timing and metrical structure of the annotated beats |

**Read Sections 1–3 first.** Each task starts with audio and a pretrained MIR specialist. The Qwen2 Audio subsection runs inference and scores the new answer. Section 4 evaluates the released test predictions.

[CMI-Bench paper](https://arxiv.org/abs/2506.12285v1) · [Official repository](https://github.com/nicolaus625/CMI-bench)


## 1 · Set up Colab and inspect the data

For the lab, open the notebook in Google Colab and choose **Runtime → Change runtime type → GPU**. Run the next two cells once to clone CMI-Bench and install the dependencies. Sections 2 and 3 run Qwen2-Audio on the test clips. The saved players below let you listen while reading. Run the audio setup cell to download the test clips from Hugging Face.

**Reading route.** Follow the notebook from the printed record to the printed answer and then to the score. Each saved output belongs to the code cell directly above it. The key example introduces how a categorical answer is graded. The beat example extends the same idea to a sequence of events, where both timing and coverage matter.


In [ ]:
!git clone https://github.com/nicolaus625/CMI-bench.git /content/CMI-bench
%cd /content/CMI-bench


Cloning into '/content/CMI-bench'...
remote: Enumerating objects: 5189, done.
remote: Counting objects: 100% (286/286), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 5189 (delta 151), reused 248 (delta 126), pack-reused 4903 (from 2)
Receiving objects: 100% (5189/5189), 79.26 MiB | 10.75 MiB/s, done.
Resolving deltas: 100% (1579/1579), done.
Updating files: 100% (13830/13830), done.
/content/CMI-bench


In [ ]:
%pip -q install transformers==4.57.3 accelerate bitsandbytes librosa mir_eval==0.8.2 \
    rouge-score nltk bert-score torchmetrics num2words jiwer pretty_midi \
    FlagEmbedding==1.3.5 sentence-transformers==5.1.2 peft==0.17.1


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


### How one example becomes a score

1. Give the audio and the task instruction to the model.
2. Read the model's text response.
3. Compare the response with the reference using the task's official evaluator.

The published Qwen2 result files already contain the prompt, response, reference and audio ID together. The cell below loads the selected example for each task, then prints the key example’s prompt, audio path and reference. These files have a `.jsonl` extension and contain JSON arrays. Set `example_index = 1` to select the second pair of clips in the demo package.


In [ ]:
import json

example_index = 0

with open("model/results/qwen2/qwen2_GS_key.jsonl") as file:
    key = json.load(file)[example_index]

with open("model/results/qwen2/qwen2_ballroom_beat.jsonl") as file:
    beat = json.load(file)[example_index]

fields = ["question", "audioid", "correct_answer"]
print(json.dumps({field: key[field] for field in fields}, indent=2, ensure_ascii=False))


{
  "question": "nstruction for Musical Key Estimation\nEstimate the musical key of the given audio. You must choose exactly one key from the following options:\n\nC major, Db major, D major, Eb major, E major, F major, Gb major, G major, Ab major, A major, Bb major, B major,\nC minor, Db minor, D minor, Eb minor, E minor, F minor, Gb minor, G minor, Ab minor, A minor, Bb minor, B minor.\n\nYour response should only contain ONE selected key.\n\nFew-Shot Examples\nExample 1:\nC major\n\nExample 2:\nE minor\n\nExample 3:\nBb major",
  "audioid": "../testdata/GS-key/giantsteps_clips/wav/0010089-0.wav",
  "correct_answer": "D major"
}


### Read the GiantSteps record

| Field | What it contains |
|---|---|
| `question` | The instruction that went to Qwen2, including the requested answer format |
| `audioid` | The path of the input audio |
| `correct_answer` | The dataset annotation used for scoring |

This example uses `0010089-0.wav`. The reference key is `D major`. The Colab lab reads the WAV file named by `audioid` and passes it to the model with `question`. The evaluator uses `correct_answer` after the model has answered.


### Set up audio for a Colab run

Run this cell to download the four test clips from the public [tutorial dataset](https://huggingface.co/datasets/Leon299/cmi-bench-tutorial). It extracts them to `/content/testdata/`, where the official result paths can find them.


In [ ]:
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

archive = hf_hub_download(
    "Leon299/cmi-bench-tutorial",
    "CMI_Bench_live_demo.zip",
    repo_type="dataset",
    token=False,
)
!unzip -qo "{archive}" "testdata/*" -d /content

key_audio = key["audioid"]
beat_audio = beat["audioid"]


CMI_Bench_live_demo.zip:   0%|          | 0.00/12.5M [00:00<?, ?B/s]

### Set up the two MIR specialists

Each task starts with a pretrained madmom model and uses the same official evaluator for madmom and Qwen2. The pinned inference runtime downloads the official madmom weights on first use. Run the audio setup above to recompute the saved specialist results.

[madmom model weights](https://github.com/CPJKU/madmom_models) · [Inference runtime](https://github.com/openmirlab/madmom-infer)


In [ ]:
%pip -q install git+https://github.com/openmirlab/madmom-infer.git@bffead9be61857fd44b2e0d3a2510d9d94c8f2d2


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from evaluate import key_ensamble_score, beat_tracking


/content/CMI-bench/evaluate.py:378: SyntaxWarning: invalid escape sequence '\('
  text = text.translate(str.maketrans('', '', '!"#$%&\()*+,-./:;<=>?@[\\]^_`{|}~'))  # remain \'


## 2 · Key detection

A musical key combines a **tonic**, the note the music centers on, with a **mode**, here major or minor. The instruction asks the model to return one label from a list of major and minor keys.

The official function `key_ensamble_score` accepts result records with a `response` and a `correct_answer`. It normalizes the response text, looks for one of the supported labels and gives that label to `mir_eval` for scoring.

The two roles of text become visible in the output below. `Live Qwen2 response` shows what the model generated from the audio and prompt. `Reference` is the annotation used by the evaluator. The evaluator parses the response into a key label before applying the scoring rule. If it cannot recover a supported key, the example contributes zero.


**Listen to the GiantSteps clip.** Notice which note feels like the tonal center before reading Qwen2's answer.

In [ ]:
display(Audio(filename=key_audio))


### Key detection with a trained CNN

The Key CNN takes a log-frequency spectrogram and predicts scores for the 24 major and minor key classes. The filters and classifier were learned during training. The label with the highest model score becomes the `response` in a copy of the example record. CMI-Bench then computes the weighted key score.

[Original madmom key model implementation](https://github.com/CPJKU/madmom/blob/main/madmom/features/key.py)

In [ ]:
from madmom_infer.audio.signal import Signal
from madmom_infer.features.key import CNNKeyRecognitionProcessor, key_prediction_to_label

key_model = CNNKeyRecognitionProcessor()
key_scores = key_model(Signal(key_audio, stop=30))
key_label = key_prediction_to_label(key_scores)

key_result = {**key, "response": key_label}
print("Reference:", key["correct_answer"])
print(f"madmom Key CNN: {key_label} · weighted key score {key_ensamble_score([key_result]):.2f}")


Reference: D major
madmom Key CNN: D major · weighted key score 1.00


### Key detection with Qwen2 Audio


Load Qwen2-Audio once in a Colab GPU runtime. The notebook uses 4-bit weights to leave GPU memory available for generation on the beat clip. The model receives the audio and the original task prompt. The helper below decodes the new answer and is also used for the beat example.

[Qwen2-Audio model card](https://huggingface.co/Qwen/Qwen2-Audio-7B-Instruct)


In [ ]:
import librosa
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2AudioForConditionalGeneration

model_id = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization,
    torch_dtype=torch.float16,
    device_map={"": 0},
).eval()


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (7 outputs left out)

model-00005-of-00005.safetensors:   0%|          | 0.00/1.28G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

In [ ]:
def ask_qwen(audio_path, question):
    sample_rate = processor.feature_extractor.sampling_rate
    audio, _ = librosa.load(audio_path, sr=sample_rate, duration=30)
    conversation = [
        {"role": "user", "content": [
            {"type": "audio", "audio": audio_path},
            {"type": "text", "text": question},
        ]}
    ]
    prompt = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )
    inputs = processor(
        text=prompt, audio=audio, sampling_rate=sample_rate, return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    answer = processor.decode(
        generated[0, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return answer


In [ ]:
key_answer = ask_qwen(key_audio, key["question"])
live_key = {**key, "response": key_answer}
print("Live Qwen2 response:", key_answer)
print("Reference:", key["correct_answer"])
print(f"Weighted key score: {key_ensamble_score([live_key]):.2f}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Live Qwen2 response: A major
Reference: D major
Weighted key score: 0.50


## 3 · Beat tracking

A beat is one pulse of the music, such as a moment where you would tap your foot. The model answers with times in seconds from the start of the clip. The evaluator parses the comma-separated timestamps and compares them with the annotated beat grid.

For **beat F1**, a prediction can match one reference beat when it falls within **70 milliseconds** of that beat. Each reference beat can be matched at most once.

- **Precision** is the fraction of predicted beats that match a reference beat.
- **Recall** is the fraction of reference beats that are matched by a prediction.
- **F1** combines them as `2 × precision × recall / (precision + recall)`.

The evaluator also reports **CMLt**, total accuracy at the annotated metrical level, and **AMLt**, the best total accuracy across allowed metrical variants such as half-time or double-time. These metrics use timing and beat-interval checks. The evaluator prints all three on a 0–100 scale.

A timestamp such as `4.77s` describes a location in the clip. The evaluator converts it to the number 4.77 before matching. A prediction at 4.77 s is 10 ms from a reference at 4.78 s, so the pair is eligible. If several predictions fall near one reference beat, one-to-one matching prevents them from all earning credit for that same beat.

The live output shows the model’s timestamp list and the first eight reference times. The scoring function receives the complete reference list. An answer that ends early can have some well-placed beats and still miss much of the reference grid.


**Listen to the Ballroom clip.** Tap along to the pulse and keep listening after 10 seconds. Compare what you hear with the timestamp list below.

In [ ]:
display(Audio(filename=beat_audio))


### Beat tracking with trained BLSTMs

The pretrained bidirectional LSTM ensemble predicts a beat activation for each 10 ms frame. An activation describes how strongly the network expects a beat at that time. A dynamic Bayesian network decoder converts the learned activations into a beat sequence with plausible timing. The code passes those timestamps to the same `beat_tracking` function used for the Qwen2 answer.

[Original madmom beat model and decoder](https://github.com/CPJKU/madmom/blob/main/madmom/features/beats.py)

In [ ]:
from madmom_infer.features.beats import RNNBeatProcessor, DBNBeatTrackingProcessor

beat_model = RNNBeatProcessor()
beat_activations = beat_model(Signal(beat_audio, stop=30))
mir_beats = DBNBeatTrackingProcessor(fps=100)(beat_activations)

beat_result = {**beat, "response": ",".join(f"{time:.4f}s" for time in mir_beats)}
print("madmom predicted beats:", len(mir_beats))
print("First eight predicted beats:", ", ".join(beat_result["response"].split(",")[:8]))
print("\nmadmom Beat BLSTM")
beat_tracking([beat_result])


madmom predicted beats: 66
First eight predicted beats: 0.2700s, 0.7100s, 1.1600s, 1.6200s, 2.0700s, 2.5300s, 2.9900s, 3.4300s

madmom Beat BLSTM
BEAT_TRACKING F1: 97.67
Average CMLt: 95.45
Average AMLt: 95.45


### Beat tracking with Qwen2 Audio


Reuse the model and helper from the key section. Compare how far the live timestamps extend and how closely they match the reference grid.


In [ ]:
beat_answer = ask_qwen(beat_audio, beat["question"])
live_beat = {**beat, "response": beat_answer}
print("Live Qwen2 response:", beat_answer)
print("Reference (first 8 beats):", ", ".join(beat["correct_answer"].split(",")[:8]))
beat_tracking([live_beat])


Live Qwen2 response: 0.00s,0.58s,1.23s,1.76s,2.33s,2.90s,3.53s,4.13s,4.77s,5.41s,6.03s,6.66s,7.28s,7.83s,8.43s,8.99s,9.55s,9.98s
Reference (first 8 beats): 1.61s, 2.07s, 2.52s, 2.97s, 3.42s, 3.87s, 4.33s, 4.78s
BEAT_TRACKING F1: 4.94
Average CMLt: 0.00
Average AMLt: 1.61


For the second pair of clips, set `example_index = 1` in Section 1. Rerun the data and audio setup, then the prediction and scoring cells for each task. Reuse the Qwen2 model already loaded in the runtime.


## 4 · From two examples to the test split

The worked scores belong to individual clips. To obtain a result for the released test predictions, run the repository's official evaluation commands below. They aggregate the published Qwen2 results separately for GiantSteps and Ballroom.

When reporting a result, give the model, dataset, metric and number of examples. A weighted key score and a beat F1 answer different questions and use different display scales.

The `--model qwen2` argument tells the script which result directory to read. The task flag chooses the matching result file and scoring function. In the output, the GiantSteps line is labelled `G-Mean` by the script. The function computes the arithmetic mean of the per-clip weighted key scores. For Ballroom, the printed F1, CMLt and AMLt values are likewise averages over the clips in the result file, displayed on a 0–100 scale.

Use the worked examples to interpret those averages. A key average can combine exact matches and partial-credit answers. An average beat F1 can combine clips with missed beats, extra beats or timing errors. Look at individual responses when you want to explain what contributed to an aggregate result.

In [ ]:
!python evaluate.py --model qwen2 --task GS_key
!python evaluate.py --model qwen2 --task ballroom_beat


/content/CMI-bench/evaluate.py:378: SyntaxWarning: invalid escape sequence '\('
  text = text.translate(str.maketrans('', '', '!"#$%&\()*+,-./:;<=>?@[\\]^_`{|}~'))  # remain \'
I0000 00:00:1794101886.941517    3595 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
sample {'question': 'nstruction for Musical Key Estimation\nEstimate the musical key of the given audio. You must choose exactly one key from the following options:\n\nC major, Db major, D major, Eb major, E major, F major, Gb major, G major, Ab major, A major, Bb major, B major,\nC minor, Db minor, D minor, Eb minor, E minor, F minor, Gb minor, G minor, Ab minor, A minor, Bb minor, B minor.\n\nYour response should only contain ONE selected key.\n\nFew-Shot Examples\nExample 1:\nC major\n\nExample 2:\nE minor\n\nExample 3:\nBb major', 'response': 'A major', 'correct_answer': 'D major', 'audioid': '../testdata/GS-key/giantsteps_clips/wav/0010089-0.wav', 'other': ''}
qwen2_GS_key G-Mean: 0.07